In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, Holt
from sklearn.metrics import mean_squared_error

In [3]:
df = pd.read_csv('C:\PROJECT DATA ANALYST\Household-Debt-Dashboard\Data Household-Debt-Dashboard/tec00104_debt_to_income.csv') 
df.head()

,DATAFLOW,LAST UPDATE,freq,unit,sector,na_item,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:TEC00104(1.0),27/07/26 11:00:00,Annual,Percentage,Households; non-profit institutions serving ho...,Gross debt-to-income ratio of households: ((AF...,Austria,2014,84.75,NaN,NaN
1,ESTAT:TEC00104(1.0),27/07/26 11:00:00,Annual,Percentage,Households; non-profit institutions serving ho...,Gross debt-to-income ratio of households: ((AF...,Austria,2015,85.46,NaN,NaN
2,ESTAT:TEC00104(1.0),27/07/26 11:00:00,Annual,Percentage,Households; non-profit institutions serving ho...,Gross debt-to-income ratio of households: ((AF...,Austria,2016,84.86,NaN,NaN
3,ESTAT:TEC00104(1.0),27/07/26 11:00:00,Annual,Percentage,Households; non-profit institutions serving ho...,Gross debt-to-income ratio of households: ((AF...,Austria,2017,84.04,NaN,NaN
4,ESTAT:TEC00104(1.0),27/07/26 11:00:00,Annual,Percentage,Households; non-profit institutions serving ho...,Gross debt-to-income ratio of households: ((AF...,Austria,2018,83.34,NaN,NaN


In [5]:
df_pl = df[df['geo'] == 'Poland'].sort_values('TIME_PERIOD').reset_index(drop=True)
df_pl

,DATAFLOW,LAST UPDATE,freq,unit,sector,na_item,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:TEC00104(1.0),27/07/26 11:00:00,Annual,Percentage,Households; non-profit institutions serving ho...,Gross debt-to-income ratio of households: ((AF...,Poland,2014,54.56,NaN,NaN
1,ESTAT:TEC00104(1.0),27/07/26 11:00:00,Annual,Percentage,Households; non-profit institutions serving ho...,Gross debt-to-income ratio of households: ((AF...,Poland,2015,56.59,NaN,NaN
2,ESTAT:TEC00104(1.0),27/07/26 11:00:00,Annual,Percentage,Households; non-profit institutions serving ho...,Gross debt-to-income ratio of households: ((AF...,Poland,2016,56.80,NaN,NaN
3,ESTAT:TEC00104(1.0),27/07/26 11:00:00,Annual,Percentage,Households; non-profit institutions serving ho...,Gross debt-to-income ratio of households: ((AF...,Poland,2017,55.06,NaN,NaN
4,ESTAT:TEC00104(1.0),27/07/26 11:00:00,Annual,Percentage,Households; non-profit institutions serving ho...,Gross debt-to-income ratio of households: ((AF...,Poland,2018,55.69,NaN,NaN
5,ESTAT:TEC00104(1.0),27/07/26 11:00:00,Annual,Percentage,Households; non-profit institutions serving ho...,Gross debt-to-income ratio of households: ((AF...,Poland,2019,55.23,NaN,NaN
6,ESTAT:TEC00104(1.0),27/07/26 11:00:00,Annual,Percentage,Households; non-profit institutions serving ho...,Gross debt-to-income ratio of households: ((AF...,Poland,2020,52.66,NaN,NaN
7,ESTAT:TEC00104(1.0),27/07/26 11:00:00,Annual,Percentage,Households; non-profit institutions serving ho...,Gross debt-to-income ratio of households: ((AF...,Poland,2021,53.55,NaN,NaN
8,ESTAT:TEC00104(1.0),27/07/26 11:00:00,Annual,Percentage,Households; non-profit institutions serving ho...,Gross debt-to-income ratio of households: ((AF...,Poland,2022,44.62,NaN,NaN
9,ESTAT:TEC00104(1.0),27/07/26 11:00:00,Annual,Percentage,Households; non-profit institutions serving ho...,Gross debt-to-income ratio of households: ((AF...,Poland,2023,39.02,NaN,NaN


In [7]:
train = df_pl['OBS_VALUE'][:-2]
test = df_pl['OBS_VALUE'][-2:]
years_test = df_pl['TIME_PERIOD'][-2:]

In [8]:
model_ses = SimpleExpSmoothing(train).fit()
forecast_ses = model_ses.forecast(2)
mse_ses = mean_squared_error(test, forecast_ses)
print("SES MSE:", mse_ses)

SES MSE: 74.60374564796423


In [9]:
model_holt = Holt(train).fit()
forecast_holt = model_holt.forecast(2)
mse_holt = mean_squared_error(test, forecast_holt)
print("Holt MSE:", mse_holt)

Holt MSE: 17.775771177290068


In [10]:
x = np.arange(len(train))
coeffs = np.polyfit(x, train, 1)
x_future = np.arange(len(train), len(train) + 2)
forecast_linear = np.polyval(coeffs, x_future)
mse_linear = mean_squared_error(test, forecast_linear)
print("Linear MSE:", mse_linear)

Linear MSE: 119.76913708178941


In [11]:
print(f"SES: {mse_ses:.3f}")
print(f"Holt: {mse_holt:.3f}")
print(f"Linear: {mse_linear:.3f}")

SES: 74.604
Holt: 17.776
Linear: 119.769


In [13]:
model_final = Holt(df_pl['OBS_VALUE']).fit()
forecast_final = model_final.forecast(3)
print(forecast_final)

11    30.733370
12    25.773409
13    20.813448
dtype: float64


In [15]:
last_year = df_pl['TIME_PERIOD'].max()
print(last_year)

2024


In [16]:
future_years = [last_year + 1, last_year + 2, last_year + 3]

df_actual = pd.DataFrame({
    'year': df_pl['TIME_PERIOD'],
    'value': df_pl['OBS_VALUE'],
    'type': 'actual'
})

df_forecast = pd.DataFrame({
    'year': future_years,
    'value': forecast_final.values,
    'type': 'forecast'
})

df_combined = pd.concat([df_actual, df_forecast], ignore_index=True)
df_combined

,year,value,type
0,2014,54.560000,actual
1,2015,56.590000,actual
2,2016,56.800000,actual
3,2017,55.060000,actual
4,2018,55.690000,actual
5,2019,55.230000,actual
6,2020,52.660000,actual
7,2021,53.550000,actual
8,2022,44.620000,actual
9,2023,39.020000,actual


In [18]:
df_combined.to_csv(r'C:\PROJECT DATA ANALYST\Household-Debt-Dashboard\Data Household-Debt-Dashboard\pl_forecast.csv', index=False)

In [25]:
residuals = model_final.resid
resid_std = residuals.std()

forecast_values = forecast_final.values
lower_bound = forecast_values - 1.96 * resid_std * np.sqrt(np.arange(1, 4))
upper_bound = forecast_values + 1.96 * resid_std * np.sqrt(np.arange(1, 4))

In [27]:
df_forecast_ci = pd.DataFrame({
    'year': future_years,
    'value': forecast_values,
    'lower': lower_bound,
    'upper': upper_bound,
    'type': 'forecast'
})

df_actual_ci = pd.DataFrame({
    'year': df_pl['TIME_PERIOD'],
    'value': df_pl['OBS_VALUE'],
    'lower': df_pl['OBS_VALUE'],
    'upper': df_pl['OBS_VALUE'],
    'type': 'actual'
})

df_combined_ci = pd.concat([df_actual_ci, df_forecast_ci], ignore_index=True)
df_combined_ci

,year,value,lower,upper,type
0,2014,54.560000,54.560000,54.560000,actual
1,2015,56.590000,56.590000,56.590000,actual
2,2016,56.800000,56.800000,56.800000,actual
3,2017,55.060000,55.060000,55.060000,actual
4,2018,55.690000,55.690000,55.690000,actual
5,2019,55.230000,55.230000,55.230000,actual
6,2020,52.660000,52.660000,52.660000,actual
7,2021,53.550000,53.550000,53.550000,actual
8,2022,44.620000,44.620000,44.620000,actual
9,2023,39.020000,39.020000,39.020000,actual


In [29]:
df_combined_ci.to_csv(r'C:\PROJECT DATA ANALYST\Household-Debt-Dashboard\Data Household-Debt-Dashboard\pl_forecast_ci.csv', index=False)